# LeetCode 1650 — Lowest Common Ancestor of a Binary Tree III bằng Python

Link đề: https://leetcode.com/problems/lowest-common-ancestor-of-a-binary-tree-iii/ · Độ khó: **Medium** · Chủ đề: Hash Table, Two Pointers, Tree, Binary Tree

> **Lưu ý (bài Premium 🔒):** đề trong notebook được viết lại theo hiểu biết về bài (nguồn tham khảo: bản mô tả của doocs/leetcode). Nếu bạn có tài khoản LeetCode Premium, hãy đối chiếu lại câu chữ, ví dụ và ràng buộc với đề gốc.

Skeleton LeetCode:

```python
"""
# Definition for a Node.
class Node:
    def __init__(self, val):
        self.val = val
        self.left = None
        self.right = None
        self.parent = None
"""

class Solution:
    def lowestCommonAncestor(self, p: 'Node', q: 'Node') -> 'Node':
        ...
```

Điểm đặc biệt: **không có gốc** trong tham số — chỉ có hai nút `p`, `q`, mỗi nút có con trỏ **`parent`**. Notebook đi từ **leo lên tìm gốc rồi giải như bài 236** → **tập tổ tiên** → **cân bằng độ sâu** → **hai con trỏ đổi đường** (giống tìm giao điểm hai danh sách liên kết, `O(1)` bộ nhớ).

## 1. Mục tiêu học

1. Nhận ra rằng đi theo `parent` từ một nút lên gốc là đi trên một **danh sách liên kết**; hai đường đi từ `p` và `q` lên gốc **hợp nhau** tại LCA.
2. Cài đặt cách **tập tổ tiên** (`set`) đơn giản, `O(h)` thời gian và bộ nhớ.
3. Cài đặt cách **cân bằng độ sâu** và cách **hai con trỏ đổi đường**, cả hai `O(1)` bộ nhớ.
4. Hiểu vì sao hai con trỏ đổi đường **chắc chắn gặp nhau** sau tối đa `d_p + d_q + 2` bước.
5. Tự dựng lớp `Node` có `parent`, dựng cây từ mảng level-order, sinh cây ngẫu nhiên và đối chiếu nhiều cách.

## 2. Đề bài & ví dụ

Cho hai nút `p` và `q` của một cây nhị phân, trả về **tổ tiên chung thấp nhất (LCA)** của chúng.

Mỗi nút có tham chiếu tới **nút cha**:

```
class Node {
    public int val;
    public Node left;
    public Node right;
    public Node parent;
}
```

Theo Wikipedia: "LCA của hai nút `p`, `q` trong cây `T` là nút thấp nhất có cả `p` và `q` là hậu duệ (cho phép một nút là hậu duệ của chính nó)."

**Ví dụ 1:** `root = [3,5,1,6,2,0,8,null,null,7,4]`, `p = 5`, `q = 1` → `3`.

**Ví dụ 2:** cùng cây, `p = 5`, `q = 4` → `5` (một nút là hậu duệ của chính nó).

**Ví dụ 3:** `root = [1,2]`, `p = 1`, `q = 2` → `1`.

**Ràng buộc:**
- Số nút trong khoảng `[2, 10⁵]`.
- `-10⁹ ≤ Node.val ≤ 10⁹`, mọi giá trị khác nhau.
- `p != q`, và `p`, `q` **chắc chắn có trong cây**.

(Mảng `root` chỉ dùng để mô tả cây; hàm của ta chỉ nhận `p`, `q`.)

In [1]:
import random
import time
from collections import deque
from typing import Optional

class Node:
    def __init__(self, val):
        self.val = val
        self.left = None
        self.right = None
        self.parent = None
    def __repr__(self):
        return f"Node({self.val})"

def build_tree(values):
    """Dựng cây có con trỏ parent từ mảng level-order kiểu LeetCode. Trả về (root, dict val -> Node)."""
    if not values:
        return None, {}
    root = Node(values[0])
    index = {root.val: root}
    queue = deque([root])
    i = 1
    while queue and i < len(values):
        node = queue.popleft()
        for side in ("left", "right"):
            if i < len(values) and values[i] is not None:
                child = Node(values[i])
                child.parent = node
                setattr(node, side, child)
                index[child.val] = child
                queue.append(child)
            i += 1
    return root, index

TREE = [3, 5, 1, 6, 2, 0, 8, None, None, 7, 4]
examples = [(TREE, 5, 1, 3), (TREE, 5, 4, 5), ([1, 2], 1, 2, 1)]
root, idx = build_tree(TREE)
for v in [7, 4, 1]:
    chain, x = [], idx[v]
    while x:
        chain.append(x.val); x = x.parent
    print(f"đi theo parent từ {v}: {chain}")

đi theo parent từ 7: [7, 2, 5, 3]
đi theo parent từ 4: [4, 2, 5, 3]
đi theo parent từ 1: [1, 3]


## 3. Kiến thức nền

### 3.1 Con trỏ `parent` biến đường lên gốc thành danh sách liên kết

Từ mọi nút, lặp `x = x.parent` sẽ tới gốc (gốc có `parent = None`). Chuỗi đó giống hệt một danh sách liên kết đơn kết thúc ở `None`. Hai chuỗi xuất phát từ `p` và `q` **dùng chung phần đuôi** bắt đầu từ LCA — giống hai danh sách liên kết **giao nhau** (LeetCode 160).

In [2]:
def chain_up(node):
    out = []
    while node:
        out.append(node.val); node = node.parent
    return out

a, b = chain_up(idx[7]), chain_up(idx[8])
print("chuỗi từ 7:", a)
print("chuỗi từ 8:", b)
print("phần đuôi chung:", [x for x in a if x in b], "-> phần tử đầu tiên là LCA")

chuỗi từ 7: [7, 2, 5, 3]
chuỗi từ 8: [8, 1, 3]
phần đuôi chung: [3] -> phần tử đầu tiên là LCA


### 3.2 `set` các đối tượng

Đối tượng `Node` mặc định băm theo **định danh** (`id`), nên có thể cho vào `set`/`dict`. `x in s` kiểm tra trong `O(1)` trung bình.

In [3]:
s = {idx[5], idx[3]}
print(idx[5] in s, idx[1] in s)
print("Node(5) mới tạo có trong set không?", Node(5) in s, "(khác đối tượng)")

True False
Node(5) mới tạo có trong set không? False (khác đối tượng)


### 3.3 So sánh `is` và `==`

Với lớp không định nghĩa `__eq__`, `a == b` **chính là** `a is b`. Hai con trỏ "gặp nhau" nghĩa là **trỏ cùng một nút**.

In [4]:
x, y = idx[5], idx[2].parent
print("x is y:", x is y, "| x == y:", x == y)
print("Node(5) == idx[5]:", Node(5) == idx[5])

x is y: True | x == y: True
Node(5) == idx[5]: False


### 3.4 Độ sâu của một nút

`depth(x)` = số bước `parent` từ `x` lên gốc. Tính bằng một vòng `while` — không cần biết gốc trước.

In [5]:
def depth(node):
    d = 0
    while node.parent:
        node = node.parent; d += 1
    return d

print({v: depth(idx[v]) for v in [3, 5, 1, 6, 2, 7, 4]})

{3: 0, 5: 1, 1: 1, 6: 2, 2: 2, 7: 3, 4: 3}


## 4. Ý tưởng nền tảng / trực giác

Tính tay với `p = 7`, `q = 8`:
- Đường từ `7` lên: `7 → 2 → 5 → 3`.
- Đường từ `8` lên: `8 → 1 → 3`.

Hai đường gặp nhau **lần đầu** ở `3` ⇒ LCA = `3`.

Với `p = 5`, `q = 4`:
- `5 → 3`; `4 → 2 → 5 → 3`. Gặp nhau lần đầu ở `5` ⇒ LCA = `5`.

Khó khăn: hai đường có **độ dài khác nhau** (`4` và `3` nút), nên nếu cho hai con trỏ cùng đi từng bước thì chúng **lệch pha**, không gặp nhau đúng lúc. Ba cách khắc phục:
1. **Nhớ** mọi nút trên đường của `p` (tập tổ tiên) rồi dò đường của `q`.
2. **Cân bằng**: đưa con trỏ sâu hơn lên trước `|d_p − d_q|` bước.
3. **Đổi đường**: con trỏ nào tới gốc thì nhảy sang điểm xuất phát của con trỏ kia — tổng quãng đường hai bên bằng nhau `d_p + d_q`, nên chúng gặp nhau đúng ở LCA.

In [6]:
for pv, qv in [(7, 8), (5, 4), (6, 4), (0, 8)]:
    a, b = chain_up(idx[pv]), chain_up(idx[qv])
    first_common = next(x for x in a if x in b)
    print(f"p={pv}: {a} | q={qv}: {b} -> gặp nhau lần đầu tại {first_common}")

p=7: [7, 2, 5, 3] | q=8: [8, 1, 3] -> gặp nhau lần đầu tại 3
p=5: [5, 3] | q=4: [4, 2, 5, 3] -> gặp nhau lần đầu tại 5
p=6: [6, 5, 3] | q=4: [4, 2, 5, 3] -> gặp nhau lần đầu tại 5
p=0: [0, 1, 3] | q=8: [8, 1, 3] -> gặp nhau lần đầu tại 1


## 5. Cách 1 — Brute Force / baseline: leo lên tìm gốc rồi giải như bài 236

**Ý tưởng:** từ `p` đi theo `parent` tới gốc. Sau đó dùng thuật toán LCA "có gốc" quen thuộc: với mỗi nút, kiểm tra cây con có chứa `p` và `q`; lấy nút sâu nhất. Đơn giản là **bỏ qua** con trỏ `parent` sau khi tìm gốc.

In [7]:
def subtree_contains(node, target):
    stack = [node]
    while stack:
        cur = stack.pop()
        if cur is target:
            return True
        if cur.left: stack.append(cur.left)
        if cur.right: stack.append(cur.right)
    return False

def lca_brute(p, q):
    r = p
    while r.parent:
        r = r.parent                      # tìm gốc
    best, best_d = None, -1
    stack = [(r, 0)]
    while stack:
        node, d = stack.pop()
        if d > best_d and subtree_contains(node, p) and subtree_contains(node, q):
            best, best_d = node, d
        if node.left: stack.append((node.left, d + 1))
        if node.right: stack.append((node.right, d + 1))
    return best

for vals, pv, qv, exp in examples:
    r, ix = build_tree(vals)
    print(f"p={pv}, q={qv} -> {lca_brute(ix[pv], ix[qv]).val} (kỳ vọng {exp})")

p=5, q=1 -> 3 (kỳ vọng 3)
p=5, q=4 -> 5 (kỳ vọng 5)
p=1, q=2 -> 1 (kỳ vọng 1)


**Độ phức tạp:** duyệt mọi nút, mỗi nút quét cây con ⇒ `O(n²)` xấu nhất (cây lệch), `O(n log n)` với cây cân bằng. Bộ nhớ `O(n)`.

**Nút thắt:** ta duyệt **cả cây** (có thể `10⁵` nút) trong khi câu trả lời chỉ nằm trên **hai đường lên gốc** dài `O(h)`. Con trỏ `parent` cho phép chỉ đi trên hai đường đó.

## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** gọi `A(x)` là tập tổ tiên của `x` (gồm chính `x`). Tổ tiên chung `A(p) ∩ A(q)` là **phần đuôi chung** của hai chuỗi `parent`, và LCA là **phần tử đầu tiên** của phần đuôi đó khi đi từ dưới lên.

**Quy tắc "nếu… thì…" của hai con trỏ đổi đường:**
- Khởi tạo `a = p`, `b = q`.
- Mỗi bước: **nếu** `a` có cha **thì** `a = a.parent`, **ngược lại** (đã ở gốc) `a = q` (nhảy sang **điểm xuất phát của bên kia**). Tương tự `b`: có cha thì lên, ở gốc thì `b = p`.
- Dừng khi `a is b`.

**Bất biến về quãng đường:** gọi `x = d_p − d_L`, `y = d_q − d_L` (số bước từ `p`, `q` lên LCA `L`), `z = d_L + 1` (số nút từ `L` tới gốc tính cả hai đầu). Con trỏ `a` đi `x + z` nút trên đường của `p`, rồi `y` bước trên đường của `q` là tới `L`; con trỏ `b` đi `y + z` rồi `x` — **cùng tổng** ⇒ gặp nhau tại `L` (hoặc sớm hơn nếu `x = y`).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Tập tổ tiên

In [8]:
def lca_set(p, q):
    seen = set()
    x = p
    while x:
        seen.add(x)                # gồm cả chính p
        x = x.parent
    y = q
    while y not in seen:
        y = y.parent
    return y

class SolutionSet:
    def lowestCommonAncestor(self, p: 'Node', q: 'Node') -> 'Node':
        seen = set()
        while p:
            seen.add(p)
            p = p.parent
        while q not in seen:
            q = q.parent
        return q

for vals, pv, qv, exp in examples:
    r, ix = build_tree(vals)
    print(f"p={pv}, q={qv} -> {lca_set(ix[pv], ix[qv]).val}, {SolutionSet().lowestCommonAncestor(ix[pv], ix[qv]).val} (kỳ vọng {exp})")

p=5, q=1 -> 3, 3 (kỳ vọng 3)
p=5, q=4 -> 5, 5 (kỳ vọng 5)
p=1, q=2 -> 1, 1 (kỳ vọng 1)


**Độ phức tạp:** `O(d_p + d_q)` = `O(h)` thời gian, `O(d_p)` = `O(h)` bộ nhớ cho `set`. Vòng thứ hai chắc chắn dừng vì gốc thuộc `seen`.

### 7.2 Cách 3 — Cân bằng độ sâu

1. Tính `dp = depth(p)`, `dq = depth(q)`.
2. Đưa nút sâu hơn lên `|dp − dq|` bước — giờ hai nút cùng độ sâu.
3. Cùng đi lên từng bước tới khi trùng nhau.

In [9]:
def lca_depth(p, q):
    dp, dq = depth(p), depth(q)
    while dp > dq:
        p = p.parent; dp -= 1
    while dq > dp:
        q = q.parent; dq -= 1
    while p is not q:
        p, q = p.parent, q.parent
    return p

class SolutionDepth:
    def lowestCommonAncestor(self, p: 'Node', q: 'Node') -> 'Node':
        def depth(x: 'Node') -> int:
            d = 0
            while x.parent:
                x = x.parent
                d += 1
            return d
        dp, dq = depth(p), depth(q)
        for _ in range(dp - dq):
            p = p.parent
        for _ in range(dq - dp):
            q = q.parent
        while p is not q:
            p, q = p.parent, q.parent
        return p

for vals, pv, qv, exp in examples:
    r, ix = build_tree(vals)
    print(f"p={pv}, q={qv} -> {lca_depth(ix[pv], ix[qv]).val}, {SolutionDepth().lowestCommonAncestor(ix[pv], ix[qv]).val} (kỳ vọng {exp})")

p=5, q=1 -> 3, 3 (kỳ vọng 3)
p=5, q=4 -> 5, 5 (kỳ vọng 5)
p=1, q=2 -> 1, 1 (kỳ vọng 1)


**Độ phức tạp:** `O(h)` thời gian (mỗi đường đi tối đa 2 lần), `O(1)` bộ nhớ.

### 7.3 Cách 4 — Hai con trỏ đổi đường (giống LeetCode 160)

In [10]:
def lca_two_pointers(p, q):
    a, b = p, q
    while a is not b:
        a = a.parent if a.parent else q
        b = b.parent if b.parent else p
    return a

for vals, pv, qv, exp in examples:
    r, ix = build_tree(vals)
    print(f"p={pv}, q={qv} -> {lca_two_pointers(ix[pv], ix[qv]).val} (kỳ vọng {exp})")

p=5, q=1 -> 3 (kỳ vọng 3)
p=5, q=4 -> 5 (kỳ vọng 5)
p=1, q=2 -> 1 (kỳ vọng 1)


**Độ phức tạp:** `O(d_p + d_q)` thời gian, `O(1)` bộ nhớ. Code ngắn nhất, nhưng cần hiểu kỹ lập luận "cùng tổng quãng đường" (mục 10).

### 7.4 Cách 5 — So sánh hai đường từ gốc xuống

Ghi hai chuỗi tổ tiên, **đảo ngược** để được đường gốc → nút, rồi lấy phần tử cuối của tiền tố chung (giống cách "hai đường đi" ở bài 1644).

In [11]:
def lca_paths(p, q):
    def path_from_root(x):
        out = []
        while x:
            out.append(x); x = x.parent
        return out[::-1]
    pa, pb = path_from_root(p), path_from_root(q)
    ans = None
    for u, v in zip(pa, pb):
        if u is not v:
            break
        ans = u
    return ans

for vals, pv, qv, exp in examples:
    r, ix = build_tree(vals)
    print(f"p={pv}, q={qv} -> {lca_paths(ix[pv], ix[qv]).val} (kỳ vọng {exp})")

p=5, q=1 -> 3 (kỳ vọng 3)
p=5, q=4 -> 5 (kỳ vọng 5)
p=1, q=2 -> 1 (kỳ vọng 1)


**Độ phức tạp:** `O(h)` thời gian và bộ nhớ.

## 8. TRACE từng bước

### 8.1 Trace cách 4 (hai con trỏ) với `p = 7`, `q = 8`

`d_7 = 3`, `d_8 = 2`. Bảng mong đợi:

| bước | a | b | ghi chú |
|---|---|---|---|
| 0 | 7 | 8 | |
| 1 | 2 | 1 | |
| 2 | 5 | 3 | b tới gốc |
| 3 | 3 | 7 | b nhảy sang p = 7 |
| 4 | 8 | 2 | a ở gốc ⇒ nhảy sang q = 8 |
| 5 | 1 | 5 | |
| 6 | 3 | 3 | **gặp nhau** ⇒ LCA = 3 |

In [12]:
def trace_two_pointers(p, q, max_steps=100):
    a, b = p, q
    step = 0
    print(f"bước {step}: a={a.val}, b={b.val}")
    while a is not b and step < max_steps:
        na = a.parent if a.parent else q
        nb = b.parent if b.parent else p
        note = []
        if a.parent is None: note.append(f"a ở gốc -> nhảy sang q={q.val}")
        if b.parent is None: note.append(f"b ở gốc -> nhảy sang p={p.val}")
        a, b = na, nb
        step += 1
        print(f"bước {step}: a={a.val}, b={b.val}  {'; '.join(note)}")
    print("=> LCA =", a.val)
    return a

r, ix = build_tree(TREE)
trace_two_pointers(ix[7], ix[8])

bước 0: a=7, b=8
bước 1: a=2, b=1  
bước 2: a=5, b=3  
bước 3: a=3, b=7  b ở gốc -> nhảy sang p=7
bước 4: a=8, b=2  a ở gốc -> nhảy sang q=8
bước 5: a=1, b=5  
bước 6: a=3, b=3  
=> LCA = 3


Node(3)

In [13]:
trace_two_pointers(ix[5], ix[4])

bước 0: a=5, b=4
bước 1: a=3, b=2  
bước 2: a=4, b=5  a ở gốc -> nhảy sang q=4
bước 3: a=2, b=3  
bước 4: a=5, b=5  b ở gốc -> nhảy sang p=5
=> LCA = 5


Node(5)

### 8.2 Trace cách 3 (cân bằng độ sâu) với `p = 4`, `q = 0`

In [14]:
def trace_depth(p, q):
    dp, dq = depth(p), depth(q)
    print(f"depth(p={p.val}) = {dp}, depth(q={q.val}) = {dq}")
    while dp > dq:
        print(f"   p sâu hơn: p {p.val} -> {p.parent.val}"); p = p.parent; dp -= 1
    while dq > dp:
        print(f"   q sâu hơn: q {q.val} -> {q.parent.val}"); q = q.parent; dq -= 1
    while p is not q:
        print(f"   cùng độ sâu {dp}: p={p.val}, q={q.val} khác nhau -> cùng lên"); p, q = p.parent, q.parent; dp -= 1
    print(f"   trùng tại {p.val} -> LCA")
    return p

trace_depth(ix[4], ix[0])

depth(p=4) = 3, depth(q=0) = 2
   p sâu hơn: p 4 -> 2
   cùng độ sâu 2: p=2, q=0 khác nhau -> cùng lên
   cùng độ sâu 1: p=5, q=1 khác nhau -> cùng lên
   trùng tại 3 -> LCA


Node(3)

## 9. Code nộp LeetCode

Lời giải chính: hai con trỏ đổi đường, `O(h)` thời gian, `O(1)` bộ nhớ.

In [15]:
"""
# Definition for a Node.
class Node:
    def __init__(self, val):
        self.val = val
        self.left = None
        self.right = None
        self.parent = None
"""

class Solution:
    def lowestCommonAncestor(self, p: 'Node', q: 'Node') -> 'Node':
        a, b = p, q
        while a is not b:
            a = a.parent if a.parent else q
            b = b.parent if b.parent else p
        return a

for vals, pv, qv, exp in examples:
    r, ix = build_tree(vals)
    assert Solution().lowestCommonAncestor(ix[pv], ix[qv]).val == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

Gọi `L` là LCA, và với mỗi nút `x` gọi `P(x)` là chuỗi nút từ `x` lên **gốc** (gồm cả hai đầu). Ta có `P(p) = U + C`, `P(q) = V + C`, trong đó `C = P(L)` là phần chung (bắt đầu tại `L`), `U` là các nút từ `p` tới ngay dưới `L` (`|U| = x`), `V` tương tự cho `q` (`|V| = y`). `U` và `V` **không có nút chung** — nếu có, nút đó là tổ tiên chung thấp hơn `L`, mâu thuẫn.

**Cách 2 (set):** `seen = U ∪ C`. Đi từ `q`: các nút trong `V` không thuộc `seen` (vì `V ∩ U = ∅`, `V ∩ C = ∅`), nút đầu tiên thuộc `seen` là `L`.

**Cách 3 (độ sâu):** `depth(p) − depth(q) = x − y`. Sau khi nâng nút sâu hơn, hai con trỏ cách `L` cùng một số bước `min(x, y)`... chính xác hơn: cùng độ sâu ⇒ tổ tiên cùng độ sâu `d_L` của chúng đều là `L`, còn ở các độ sâu lớn hơn `d_L` chúng thuộc `U`, `V` rời nhau ⇒ không trùng. Vòng lặp dừng đúng tại `L`.

**Cách 4 (hai con trỏ):** con trỏ `a` đi dãy `U, C, V, C, …` (khi tới gốc thì nhảy sang `q`, tức đầu `V`); con trỏ `b` đi dãy `V, C, U, C, …`. Sau `x + |C| + y` bước, `a` đứng tại phần tử đầu của `C` lần thứ hai, tức `L`; `b` sau `y + |C| + x` bước cũng đứng tại `L`. Vì cùng số bước, hai con trỏ **gặp nhau muộn nhất** tại đó. Chúng không thể "gặp nhầm" sớm hơn ở một nút khác `L`: nếu gặp tại nút `w` thì `w` nằm trên cả hai chuỗi ⇒ `w ∈ C`; lần đầu cả hai cùng ở trong `C` tại cùng vị trí chỉ có thể là `L` (nếu `x = y` thì gặp ngay lần đầu tới `L`). Vì `p`, `q` chắc chắn cùng cây nên `C` khác rỗng — vòng lặp luôn dừng.

## 11. Phân tích Time / Space Complexity

Gọi `n` là số nút, `h` là chiều cao cây, `d_p`, `d_q` là độ sâu của `p`, `q`.

| Cách | Time | Space |
|---|---|---|
| 1. Tìm gốc + duyệt cả cây | `O(n²)` xấu nhất | `O(n)` |
| 2. Tập tổ tiên | `O(d_p + d_q)` | `O(d_p)` |
| 3. Cân bằng độ sâu | `O(d_p + d_q)` | `O(1)` |
| 4. Hai con trỏ đổi đường | `O(d_p + d_q)` | `O(1)` |
| 5. Hai đường từ gốc | `O(d_p + d_q)` | `O(d_p + d_q)` |

Tất cả trừ cách 1 đều **không phụ thuộc `n`**, chỉ phụ thuộc độ sâu. Đo thời gian trên cây lệch `10⁵` nút (độ sâu tới `10⁵`) — không có đệ quy nên không lo giới hạn đệ quy:

In [16]:
def skewed(n):
    root = Node(0); cur = root
    nodes = [root]
    for v in range(1, n):
        nxt = Node(v); nxt.parent = cur
        if v % 2: cur.left = nxt
        else: cur.right = nxt
        cur = nxt; nodes.append(cur)
    return root, nodes

R, nodes = skewed(10**5)
# nhánh phụ để p, q ở hai nhánh khác nhau
extra = Node(-1); extra.parent = nodes[50000]
if nodes[50000].left is None: nodes[50000].left = extra
else: nodes[50000].right = extra
for name, fn in [("set", lca_set), ("depth", lca_depth), ("two_ptr", lca_two_pointers), ("paths", lca_paths)]:
    t0 = time.perf_counter(); a = fn(nodes[-1], extra); dt = time.perf_counter() - t0
    print(f"{name:<8}: LCA={a.val}  {dt * 1000:6.1f} ms")

set     : LCA=50000     7.1 ms
depth   : LCA=50000     5.0 ms
two_ptr : LCA=50000     3.7 ms
paths   : LCA=50000     5.6 ms


## 12. Bộ test

Đối chiếu tất cả các cách với brute force trên cây ngẫu nhiên nhỏ (mọi cặp `p ≠ q`), ví dụ đề, edge case (hai nút, `p` là gốc, `p` là tổ tiên của `q` và ngược lại, hai lá xa nhau) và cây lệch lớn.

In [17]:
def random_tree(n, rnd):
    vals = rnd.sample(range(-10**9, 10**9), n)
    nodes = [Node(vals[0])]
    slots = [(nodes[0], 'left'), (nodes[0], 'right')]
    for v in vals[1:]:
        k = rnd.randrange(len(slots))
        par, side = slots[k]
        slots[k] = slots[-1]; slots.pop()
        ch = Node(v); ch.parent = par; setattr(par, side, ch)
        nodes.append(ch)
        slots += [(ch, 'left'), (ch, 'right')]
    return nodes[0], nodes

ALL = {
    "brute": lca_brute,
    "set": lca_set,
    "depth": lca_depth,
    "two_ptr": lca_two_pointers,
    "paths": lca_paths,
    "SolSet": lambda p, q: SolutionSet().lowestCommonAncestor(p, q),
    "SolDepth": lambda p, q: SolutionDepth().lowestCommonAncestor(p, q),
    "Solution": lambda p, q: Solution().lowestCommonAncestor(p, q),
}

def check(p, q, exp=None):
    res = {name: fn(p, q) for name, fn in ALL.items()}
    first = res["brute"]
    assert all(v is first for v in res.values()), {k: v.val for k, v in res.items()}
    if exp is not None:
        assert first.val == exp
    return first

for vals, pv, qv, exp in examples:
    r, ix = build_tree(vals)
    check(ix[pv], ix[qv], exp)
    check(ix[qv], ix[pv], exp)          # đổi vai trò p, q
print("Ví dụ đề (cả hai chiều): OK")

r, ix = build_tree(TREE)
check(ix[3], ix[4], 3); check(ix[2], ix[7], 2); check(ix[6], ix[8], 3); check(ix[7], ix[4], 2); check(ix[0], ix[8], 1)
r, ix = build_tree([1, None, 2, None, 3, None, 4])
check(ix[4], ix[2], 2); check(ix[1], ix[4], 1)
print("Edge case: OK")

Ví dụ đề (cả hai chiều): OK
Edge case: OK


In [18]:
rnd = random.Random(1650)
cnt = 0
for _ in range(300):
    n = rnd.randint(2, 12)
    r, nodes = random_tree(n, rnd)
    for p in nodes:
        for q in nodes:
            if p is not q:
                check(p, q); cnt += 1
print(f"{cnt} cặp (p, q) trên 300 cây ngẫu nhiên nhỏ: OK")

for _ in range(20):
    r, nodes = random_tree(rnd.randint(1000, 3000), rnd)
    p, q = rnd.sample(nodes, 2)
    base = lca_set(p, q)
    assert lca_depth(p, q) is base and lca_two_pointers(p, q) is base and Solution().lowestCommonAncestor(p, q) is base
print("20 cây cỡ vừa (bỏ brute force): OK")

15216 cặp (p, q) trên 300 cây ngẫu nhiên nhỏ: OK


20 cây cỡ vừa (bỏ brute force): OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Hai con trỏ nhảy về **chính điểm xuất phát của mình** khi tới gốc

In [19]:
def lca_bug_restart_self(p, q, max_steps=10**5):
    a, b = p, q
    steps = 0
    while a is not b and steps < max_steps:      # giới hạn bước chỉ để demo không treo
        a = a.parent if a.parent else p          # SAI: phải nhảy sang q
        b = b.parent if b.parent else q          # SAI: phải nhảy sang p
        steps += 1
    return (a if a is b else None), steps

def count_steps_correct(p, q):
    a, b, steps = p, q, 0
    while a is not b:
        a = a.parent if a.parent else q
        b = b.parent if b.parent else p
        steps += 1
    return steps

def two_branches(x, y):
    """Gốc có hai nhánh thẳng dài x và y; trả về (lá nhánh 1, lá nhánh 2)."""
    root = Node(0); ends = []
    for side, length, base in (("left", x, 1), ("right", y, 10**6)):
        cur = root
        for k in range(length):
            nxt = Node(base + k); nxt.parent = cur; setattr(cur, side, nxt); cur = nxt
        ends.append(cur)
    return ends

r, ix = build_tree(TREE)
res, steps = lca_bug_restart_self(ix[7], ix[8])
print("Cây nhỏ (7, 8): bản sai gặp tại", res, "sau", steps, "bước | bản đúng:", count_steps_correct(ix[7], ix[8]), "bước")
p_leaf, q_leaf = two_branches(500, 499)
res, steps = lca_bug_restart_self(p_leaf, q_leaf)
print("Hai nhánh dài 500 và 499: bản sai sau", steps, "bước ->", "chưa gặp!" if res is None else res,
      "| bản đúng:", count_steps_correct(p_leaf, q_leaf), "bước ->", Solution().lowestCommonAncestor(p_leaf, q_leaf))

Cây nhỏ (7, 8): bản sai gặp tại Node(3) sau 11 bước | bản đúng: 6 bước
Hai nhánh dài 500 và 499: bản sai sau 100000 bước -> chưa gặp! | bản đúng: 1000 bước -> Node(0)


**Vì sao sai:** chu kỳ của `a` là `d_p + 1` nút, của `b` là `d_q + 1` nút. Hai con trỏ chỉ trùng nhau khi số bước `t` thỏa **đồng thời** hai đồng dư — tức phải chờ tới cỡ **bội chung nhỏ nhất** `lcm(d_p + 1, d_q + 1)` bước. Trên cây nhỏ vẫn "may mắn" ra đúng (nhưng tốn 11 bước thay vì 6); với hai nhánh dài 500 và 499 phải chờ `501 · 500 ≈ 2.5·10⁵` bước, còn cây `10⁵` nút có thể lên tới `~10¹⁰` bước ⇒ **Time Limit Exceeded**. Mấu chốt của mẹo đúng là mỗi con trỏ đi **cả hai đường** để tổng quãng đường bằng nhau `d_p + d_q + 2`. **Sửa:** `a` nhảy sang `q`, `b` nhảy sang `p`.

### Lỗi 2 — Không cân bằng độ sâu mà cho hai con trỏ đi lên cùng lúc

In [20]:
def lca_bug_no_balance(p, q):
    while p is not q and p.parent and q.parent:
        p, q = p.parent, q.parent           # SAI: lệch pha nếu độ sâu khác nhau
    return p if p is q else (p if p.parent is None else q)   # "đoán" khi một bên chạm gốc

r, ix = build_tree(TREE)
print("không cân bằng (p=5, q=4):", lca_bug_no_balance(ix[5], ix[4]), "| đúng:", Solution().lowestCommonAncestor(ix[5], ix[4]))
print("không cân bằng (p=7, q=0):", lca_bug_no_balance(ix[7], ix[0]), "| đúng:", Solution().lowestCommonAncestor(ix[7], ix[0]))

không cân bằng (p=5, q=4): Node(3) | đúng: Node(5)
không cân bằng (p=7, q=0): Node(3) | đúng: Node(3)


**Vì sao sai:** `5` ở độ sâu 1, `4` ở độ sâu 3. Đi lên cùng lúc: `(5, 4) → (3, 2)` — `5` đã bị vượt qua mà không bao giờ trùng với con trỏ kia. **Sửa:** nâng nút sâu hơn lên `|d_p − d_q|` bước trước (cách 3) hoặc dùng đổi đường (cách 4).

### Lỗi 3 — Tập tổ tiên không chứa chính `p`

In [21]:
def lca_bug_set_excl(p, q):
    seen = set()
    x = p.parent                      # SAI: bỏ qua chính p
    while x:
        seen.add(x); x = x.parent
    y = q
    while y not in seen:
        y = y.parent
    return y

r, ix = build_tree(TREE)
print("tập không chứa p (p=5, q=4):", lca_bug_set_excl(ix[5], ix[4]), "| đúng:", Solution().lowestCommonAncestor(ix[5], ix[4]))

tập không chứa p (p=5, q=4): Node(3) | đúng: Node(5)


**Vì sao sai:** đề cho phép một nút là hậu duệ của chính nó; khi `p` là tổ tiên của `q`, đáp án là `p`. **Sửa:** khởi tạo `x = p`.

### Lỗi 4 — So sánh bằng `val` giữa hai cây khác nhau / dùng `==` sau khi định nghĩa `__eq__` theo `val`

Nếu bạn (hoặc thư viện) định nghĩa `Node.__eq__` so sánh theo `val` nhưng **không** định nghĩa `__hash__`, đối tượng trở nên **không băm được** và `set` báo lỗi. Ta minh họa mà không để lỗi văng ra:

In [22]:
class NodeEq(Node):
    def __eq__(self, other):
        return isinstance(other, Node) and self.val == other.val

a = NodeEq(1)
try:
    s = {a}
    print("băm được")
except TypeError as e:
    print("TypeError:", e)
print("Bài học: so sánh con trỏ bằng `is`, đừng phụ thuộc __eq__ tùy biến.")

TypeError: unhashable type: 'NodeEq'
Bài học: so sánh con trỏ bằng `is`, đừng phụ thuộc __eq__ tùy biến.


**Sửa:** dùng `is`/`is not` khi so sánh nút (hai con trỏ gặp nhau = cùng đối tượng); `set` dựa trên định danh mặc định.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Có con trỏ `parent` ⇒ đường lên gốc là **danh sách liên kết** ⇒ LCA = **giao điểm hai danh sách** (LeetCode 160).
- Ba khuôn: `set` (dễ), **cân bằng độ dài** (dễ chứng minh), **đổi đường** (ngắn nhất, `O(1)`).
- Không có gốc trong tham số không phải là trở ngại: leo `parent` là tới gốc.
- Không dùng đệ quy cho bài có `10⁵` nút trong Python — bài này vốn dĩ lặp nên rất an toàn.
- Khi làm "đổi đường", tự hỏi: mỗi con trỏ đã đi **đúng** `d_p + d_q + 2` nút chưa?

## 15. Biến thể của bài

1. **LeetCode 160 — Intersection of Two Linked Lists:** cùng mẹo hai con trỏ đổi đường.
2. **LeetCode 236 / 1644 / 1676:** có gốc, không có `parent` ⇒ DFS hậu thứ tự.
3. **Khoảng cách giữa `p` và `q`:** `d_p + d_q − 2·d_L` — tính luôn trong cách cân bằng độ sâu.
4. **LCA của `k` nút có `parent`:** tính LCA từng cặp cộng dồn, hoặc đếm số lần mỗi tổ tiên được thăm.
5. **Nhiều truy vấn LCA:** nhảy nhị phân (`up[j][x]` = tổ tiên thứ `2^j`), mỗi truy vấn `O(log n)`.
6. **Đồ thị có hướng không chu trình (DAG)** — "tổ tiên chung" có thể nhiều; cần BFS từ hai phía.

In [23]:
def distance(p, q):
    L = Solution().lowestCommonAncestor(p, q)
    return depth(p) + depth(q) - 2 * depth(L)

def lca_many(nodes):
    cur = nodes[0]
    for x in nodes[1:]:
        cur = Solution().lowestCommonAncestor(cur, x)
    return cur

r, ix = build_tree(TREE)
print("d(7, 8) =", distance(ix[7], ix[8]), "| d(5, 4) =", distance(ix[5], ix[4]))
print("LCA(7, 4, 6) =", lca_many([ix[7], ix[4], ix[6]]).val, "| LCA(7, 0) =", lca_many([ix[7], ix[0]]).val)

d(7, 8) = 5 | d(5, 4) = 2
LCA(7, 4, 6) = 5 | LCA(7, 0) = 3


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Tìm gốc + 236 | bỏ qua `parent`, duyệt cả cây | `O(n²)` / `O(n)` | `O(n)` | đối chiếu |
| 2. Tập tổ tiên | `set` đường của `p`, dò đường `q` | `O(h)` | `O(h)` | nhanh viết, ít lỗi |
| 3. Cân bằng độ sâu | nâng nút sâu rồi cùng lên | `O(h)` | `O(1)` | dễ chứng minh, `O(1)` |
| 4. Đổi đường | `a: p→gốc→q→…`, `b: q→gốc→p→…` | `O(h)` | `O(1)` | **nộp bài**, ngắn nhất |
| 5. Hai đường từ gốc | tiền tố chung | `O(h)` | `O(h)` | khi cần cả đường đi |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** Trên cây ví dụ, LCA của `(6, 7)`, `(0, 3)` và `(2, 4)` là gì?

**Bài 2 (tính tay):** Trace cách 4 (hai con trỏ) với `p = 6`, `q = 0`: liệt kê `(a, b)` qua từng bước, cho biết số bước tới khi gặp.

**Bài 3 (điền TODO):** Hoàn thiện `path_between(p, q)` trả về danh sách giá trị trên đường đi từ `p` tới `q` (qua LCA).

**Bài 4 (tự viết lại):** Viết lại cách 3 sao cho hàm trả về **đồng thời** LCA và khoảng cách giữa `p`, `q`, chỉ với `O(1)` bộ nhớ.

**Bài 5 (mini challenge):** Cài đặt **nhảy nhị phân** (binary lifting) cho cây có `parent`: tiền xử lý `up[j][x]` cho mọi nút rồi trả lời nhiều truy vấn LCA trong `O(log n)` mỗi truy vấn. Kiểm với `Solution` trên cây ngẫu nhiên.

In [24]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def path_between(p, q):
    # TODO: L = LCA; đi từ p lên L (gồm L), đi từ q lên ngay dưới L, đảo ngược phần của q rồi nối
    pass

# Bài 4 — TODO
def lca_and_distance(p, q):
    # TODO: tính depth, nâng nút sâu (đếm bước), cùng lên (đếm 2 bước mỗi lần)
    pass

# Bài 5 — TODO
def build_lifting(nodes):
    # TODO: depth[x], up[0][x] = x.parent; up[j][x] = up[j-1][up[j-1][x]]
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: `6` và `7` đều dưới `5`.
- Mức 2: `3` là gốc — là tổ tiên của mọi nút.
- Mức 3: `2` là cha của `4`.

**Bài 2**
- Mức 1: `d_6 = 2`, `d_0 = 2` — cùng độ sâu.
- Mức 2: Hai con trỏ cùng tới gốc ở cùng một bước.
- Mức 3: Gặp nhau ngay khi cả hai tới `3` (bước 2), không cần đổi đường.

**Bài 3**
- Mức 1: Đường `p → L` đi theo `parent`.
- Mức 2: Đường `L → q` là đảo ngược của `q → L` (bỏ `L` để không lặp).
- Mức 3: Kiểm độ dài đường = khoảng cách + 1.

**Bài 4**
- Mức 1: Khoảng cách = `|d_p − d_q|` + 2 × (số bước cùng lên).
- Mức 2: Đếm trong các vòng `while`.
- Mức 3: So với `distance()` ở mục 15.

**Bài 5**
- Mức 1: Gốc có `up[0][root] = root` (tự trỏ) để tránh `None`.
- Mức 2: Nâng nút sâu hơn bằng các bit của hiệu độ sâu.
- Mức 3: Với `j` từ lớn xuống nhỏ: nếu `up[j][a] is not up[j][b]` thì nhảy cả hai; cuối cùng trả `up[0][a]`.

## 19. Đáp án tham khảo cho bài tập

In [25]:
# Bài 1
r, ix = build_tree(TREE)
res1 = [Solution().lowestCommonAncestor(ix[a], ix[b]).val for a, b in [(6, 7), (0, 3), (2, 4)]]
print("Bài 1:", res1)
assert res1 == [5, 3, 2]

# Bài 2
ans2 = trace_two_pointers(ix[6], ix[0])
assert ans2.val == 3

Bài 1: [5, 3, 2]
bước 0: a=6, b=0
bước 1: a=5, b=1  
bước 2: a=3, b=3  
=> LCA = 3


In [26]:
# Bài 3
def path_between_sol(p, q):
    L = Solution().lowestCommonAncestor(p, q)
    left = []
    x = p
    while x is not L:
        left.append(x.val); x = x.parent
    left.append(L.val)
    right = []
    y = q
    while y is not L:
        right.append(y.val); y = y.parent
    return left + right[::-1]

# Bài 4
def lca_and_distance_sol(p, q):
    dp, dq = depth(p), depth(q)
    dist = 0
    while dp > dq:
        p = p.parent; dp -= 1; dist += 1
    while dq > dp:
        q = q.parent; dq -= 1; dist += 1
    while p is not q:
        p, q = p.parent, q.parent; dist += 2
    return p, dist

r, ix = build_tree(TREE)
print("Bài 3: đường 7 -> 8:", path_between_sol(ix[7], ix[8]), "| 4 -> 5:", path_between_sol(ix[4], ix[5]))
assert path_between_sol(ix[7], ix[8]) == [7, 2, 5, 3, 1, 8]
rnd = random.Random(4)
for _ in range(300):
    rt, nodes = random_tree(rnd.randint(2, 30), rnd)
    p, q = rnd.sample(nodes, 2)
    L, d = lca_and_distance_sol(p, q)
    assert L is Solution().lowestCommonAncestor(p, q) and d == distance(p, q) == len(path_between_sol(p, q)) - 1
print("Bài 4 OK")

Bài 3: đường 7 -> 8: [7, 2, 5, 3, 1, 8] | 4 -> 5: [4, 2, 5]
Bài 4 OK


In [27]:
# Bài 5
def build_lifting_sol(nodes):
    LOG = max(1, len(nodes).bit_length())
    root = nodes[0]
    while root.parent:
        root = root.parent
    dep = {root: 0}
    order = deque([root])
    while order:                                   # BFS để có độ sâu
        x = order.popleft()
        for ch in (x.left, x.right):
            if ch:
                dep[ch] = dep[x] + 1; order.append(ch)
    up = [{x: (x.parent or x) for x in nodes}]
    for j in range(1, LOG):
        prev = up[j - 1]
        up.append({x: prev[prev[x]] for x in nodes})
    def query(a, b):
        if dep[a] < dep[b]:
            a, b = b, a
        diff = dep[a] - dep[b]
        j = 0
        while diff:
            if diff & 1:
                a = up[j][a]
            diff >>= 1; j += 1
        if a is b:
            return a
        for j in range(LOG - 1, -1, -1):
            if up[j][a] is not up[j][b]:
                a, b = up[j][a], up[j][b]
        return up[0][a]
    return query

rnd = random.Random(5)
for _ in range(30):
    rt, nodes = random_tree(rnd.randint(2, 400), rnd)
    query = build_lifting_sol(nodes)
    for _ in range(100):
        p, q = rnd.sample(nodes, 2)
        assert query(p, q) is Solution().lowestCommonAncestor(p, q)
r, ix = build_tree(TREE)
query = build_lifting_sol(list(ix.values()))
print("Bài 5 OK | query(7, 8) =", query(ix[7], ix[8]).val, "| query(5, 4) =", query(ix[5], ix[4]).val)

Bài 5 OK | query(7, 8) = 3 | query(5, 4) = 5


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template hai con trỏ đổi đường (giao điểm hai "danh sách" lên gốc):**

```python
a, b = p, q
while a is not b:
    a = a.parent if a.parent else q     # tới gốc thì sang điểm xuất phát của bên kia
    b = b.parent if b.parent else p
return a
```

**Template cân bằng độ sâu:**

```python
dp, dq = depth(p), depth(q)
for _ in range(dp - dq): p = p.parent
for _ in range(dq - dp): q = q.parent
while p is not q: p, q = p.parent, q.parent
return p
```

**Checklist:**
- [ ] Khi tới gốc, con trỏ nhảy sang điểm xuất phát của **bên kia** (không phải của chính mình)?
- [ ] Tập tổ tiên có chứa **chính `p`**?
- [ ] Cân bằng độ sâu trước khi cùng đi lên?
- [ ] So sánh nút bằng `is`?
- [ ] Không đệ quy (cây lệch tới `10⁵`)?
- [ ] Đã kiểm mọi cặp `(p, q)` trên cây ngẫu nhiên nhỏ, cả hai chiều?
- [ ] Chữ ký đúng: `def lowestCommonAncestor(self, p: 'Node', q: 'Node') -> 'Node'` (không có `root`)?